# 01 — Exploración de la API del BCRA

Objetivo: confirmar qué series están disponibles en la API pública del BCRA y bajar las primeras series relevantes para el proyecto (tasas, mora agregada si existiera).

Dos APIs distintas:

- **Principales Variables (v4.0)**: series monetarias/financieras agregadas.
- **Central de Deudores (v1.0)**: consulta por CUIT/CUIL/CDI puntual — **no** trae mora agregada por entidad/cartera.

La mora agregada por entidad y tipo de cartera (consumo/comercial) no tiene API JSON: se publica como Excel en el [Anexo estadístico del Informe sobre Bancos](https://www.bcra.gob.ar/catalogo_de_datos/anexo-estadistico-del-informe-sobre-bancos/). Ese archivo se descarga manualmente y se guarda en `data/raw/`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bcra_api import list_monetary_variables, get_monetary_series

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

## Catálogo de variables monetarias

Si `requests` tira `SSLError` por la cadena de certificados incompleta del lado del BCRA, reintentar con `verify=False` (ver docstring de `src/bcra_api.py`).

In [ ]:
try:
    variables = list_monetary_variables()
except Exception as exc:
    print(f"Fallo con verify=True ({exc}); reintentando con verify=False")
    variables = list_monetary_variables(verify=False)

df_variables = pd.DataFrame(variables)
df_variables.to_csv(RAW_DIR / "bcra_variables_monetarias.csv", index=False)
df_variables.head(20)

## Buscar series relevantes (tasas, cartera irregular)

Filtramos por descripción para encontrar los `idVariable` de interés (tasa de política monetaria, tasas activas, etc.).

In [ ]:
keywords = ["tasa", "cartera", "irregular", "mora", "lefi"]
mask = df_variables["descripcion"].str.lower().str.contains("|".join(keywords), na=False)
df_variables[mask]

## Descargar una serie de ejemplo

`id_variable=144` es la tasa de interés de préstamos personales otorgados al sector privado — el proxy más directo de lo que efectivamente paga un deudor minorista.

In [ ]:
id_variable = 144  # Tasa de interés de préstamos personales otorgados al sector privado
serie = get_monetary_series(id_variable, desde="2023-01-01", verify=False)
df_serie = pd.DataFrame(serie)
df_serie.to_csv(RAW_DIR / f"bcra_serie_{id_variable}.csv", index=False)
df_serie.head()

## Chequeo: ¿hay un quiebre de tasas alrededor del fin de las LEFIs (jul-2025)?

Ver `docs/planteo_catedra.md`, sección 2.4, para el argumento completo. Acá está el cálculo que sostiene esa sección: BADLAR (id 7), tasa de préstamos personales (id 144), stock de LEFIs (id 196, confirma el vencimiento exacto) y tipo de cambio minorista (id 4, para descartar que el salto sea solo ruido pre-electoral — elecciones legislativas 26/10/2025).

In [ ]:
def load_series(id_var, desde, hasta):
    data = get_monetary_series(id_var, desde=desde, hasta=hasta, verify=False)
    df = pd.DataFrame(data)
    df["fecha"] = pd.to_datetime(df["fecha"])
    return df.sort_values("fecha").set_index("fecha")["valor"]


badlar = load_series(7, "2025-01-01", "2025-10-31")
personales = load_series(144, "2025-01-01", "2025-10-31")
lefi_stock = load_series(196, "2025-01-01", "2025-10-31")
usd_minorista = load_series(4, "2025-01-01", "2025-10-31")

print("Último día con stock de LEFIs > 0:", lefi_stock[lefi_stock > 0].index.max().date())

In [ ]:
windows = {
    "pre (abr-jun 2025)": ("2025-04-01", "2025-06-30"),
    "transicion (jul 2025)": ("2025-07-01", "2025-07-31"),
    "post inmediato (ago 2025)": ("2025-08-01", "2025-08-31"),
    "post (sep 2025)": ("2025-09-01", "2025-09-30"),
}

resumen = pd.DataFrame(
    {
        "BADLAR": {name: badlar[a:b].mean() for name, (a, b) in windows.items()},
        "tasa_prestamos_personales": {name: personales[a:b].mean() for name, (a, b) in windows.items()},
        "usd_minorista": {name: usd_minorista[a:b].mean() for name, (a, b) in windows.items()},
    }
)
resumen

**Lectura (ver detalle en `docs/planteo_catedra.md` 2.4):** no hay quiebre inmediato el 10/07/2025 — BADLAR y la tasa de préstamos personales bajan levemente en julio. El salto grande (BADLAR +48% relativo, préstamos personales +13%) llega recién en agosto-septiembre, con ~1 mes de rezago respecto del vencimiento de LEFIs. El dólar, en cambio, sube más en julio y se desacelera en agosto — es decir, tasas y tipo de cambio se desacoplan mes a mes, lo que pesa en contra de explicar el salto de tasas solo por nerviosismo pre-electoral (elecciones 26/10/2025) y deja viva, aunque no confirmada, la hipótesis de policy feedback vía LEFIs. Pendiente: un evento-estudio con datos diarios en vez de promedios mensuales, y mirar el spread de BADLAR sobre la tasa de política monetaria.

## Próximo paso: mora agregada por entidad

1. Descargar manualmente el Excel del [Anexo estadístico del Informe sobre Bancos](https://www.bcra.gob.ar/catalogo_de_datos/anexo-estadistico-del-informe-sobre-bancos/) y guardarlo en `data/raw/`.
2. Escribir un parser en `src/clean.py` para normalizar las hojas de irregularidad de cartera por grupo de entidades.
3. Continuar en `02_mora_bancos_vs_fintech.ipynb`.